In [22]:
import sqlite3
import pandas as pd
import os
import openpyxl

In [23]:
DB_PATH   = 'final.db'
DATA_PATH = 'online_retail_II.xlsx'

In [ ]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.executescript("""

CREATE TABLE Customer (
    customer_id INTEGER PRIMARY KEY,
    country     TEXT    NOT NULL
);

CREATE TABLE Product (
    stock_code  TEXT PRIMARY KEY,
    description TEXT NOT NULL,
    unit_price  REAL NOT NULL CHECK (unit_price >= 0)
);

CREATE TABLE Invoice (
    invoice_id      TEXT    PRIMARY KEY,
    invoice_date    TEXT    NOT NULL,
    customer_id     INTEGER NOT NULL,
    is_cancellation INTEGER NOT NULL DEFAULT 0
                            CHECK (is_cancellation IN (0, 1)),
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id)
);

CREATE TABLE InvoiceLineItem (
    line_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id   TEXT    NOT NULL,
    stock_code   TEXT    NOT NULL,
    quantity     INTEGER NOT NULL,
    unit_price   REAL    NOT NULL CHECK (unit_price >= 0),
    line_total   REAL    NOT NULL,
    FOREIGN KEY (invoice_id) REFERENCES Invoice(invoice_id),
    FOREIGN KEY (stock_code) REFERENCES Product(stock_code)
);

""")

conn.commit()

Removed existing final.db


In [25]:
sheets = pd.read_excel(DATA_PATH, sheet_name=None, dtype={'Customer ID': 'Int64'})
df = pd.concat(sheets.values(), ignore_index=True)
print(f"Raw rows loaded:     {len(df):,}")

df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)

df['is_cancellation'] = df['Invoice'].astype(str).str.startswith('C').astype(int)

df = df[df['Price'] > 0]

SPECIAL_CODES = {'POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK', 'C2', 'SP1002'}
df = df[~df['StockCode'].isin(SPECIAL_CODES)]
df = df[df['StockCode'].astype(str).str.match(r'^[A-Za-z0-9]+$', na=False)]

df['Description'] = df['Description'].astype(str).str.strip().str.upper()
df = df[df['Description'].notna() & (df['Description'] != 'NAN')]

df['LineTotal'] = df['Quantity'] * df['Price']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

print(f"Rows after cleaning: {len(df):,}")
print(f"Unique customers:    {df['Customer ID'].nunique():,}")
print(f"Unique products:     {df['StockCode'].nunique():,}")
print(f"Unique invoices:     {df['Invoice'].nunique():,}")

Raw rows loaded:     1,067,371
Rows after cleaning: 820,643
Unique customers:    5,894
Unique products:     4,637
Unique invoices:     43,954


In [26]:
customers = (
    df[['Customer ID', 'Country']]
    .drop_duplicates(subset='Customer ID')
    .rename(columns={'Customer ID': 'customer_id', 'Country': 'country'})
)
customers.to_sql('Customer', conn, if_exists='append', index=False)
print(f"Customers inserted:  {len(customers):,}")

# One row per stock_code: most common description, median unit price
products = (
    df.groupby('StockCode')
      .agg(description=('Description', lambda x: x.mode()[0]),
           unit_price=('Price', 'median'))
      .reset_index()
      .rename(columns={'StockCode': 'stock_code'})
)
products.to_sql('Product', conn, if_exists='append', index=False)
print(f"Products inserted:   {len(products):,}")

invoices = (
    df[['Invoice', 'InvoiceDate', 'Customer ID', 'is_cancellation']]
    .drop_duplicates(subset='Invoice')
    .rename(columns={'Invoice': 'invoice_id',
                     'InvoiceDate': 'invoice_date',
                     'Customer ID': 'customer_id'})
)
invoices.to_sql('Invoice', conn, if_exists='append', index=False)
print(f"Invoices inserted:   {len(invoices):,}")

line_items = (
    df[['Invoice', 'StockCode', 'Quantity', 'Price', 'LineTotal']]
    .rename(columns={'Invoice': 'invoice_id',
                     'StockCode': 'stock_code',
                     'Quantity': 'quantity',
                     'Price': 'unit_price',
                     'LineTotal': 'line_total'})
)
line_items.to_sql('InvoiceLineItem', conn, if_exists='append', index=False)
print(f"Line items inserted: {len(line_items):,}")

conn.commit()

Customers inserted:  5,894
Products inserted:   4,637
Invoices inserted:   43,954
Line items inserted: 820,643


In [27]:
print("Row counts by table:")
for table in ['Customer', 'Product', 'Invoice', 'InvoiceLineItem']:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    print(f"  {table:<22} {n:>10,}")

Row counts by table:
  Customer                    5,894
  Product                     4,637
  Invoice                    43,954
  InvoiceLineItem           820,643


In [28]:
q1 = pd.read_sql_query("""
    SELECT ROUND(SUM(li.line_total), 2) AS total_revenue_gbp
    FROM   InvoiceLineItem li
    JOIN   Invoice i ON li.invoice_id = i.invoice_id
    WHERE  i.is_cancellation = 0
""", conn)
print("Total Revenue (GBP):")
q1

Total Revenue (GBP):


,total_revenue_gbp
0,17438945.55


In [29]:
q2 = pd.read_sql_query("""
    SELECT
        COUNT(DISTINCT i.invoice_id)                                    AS total_orders,
        ROUND(SUM(li.line_total) / COUNT(DISTINCT i.invoice_id), 2)     AS avg_order_value_gbp
    FROM   Invoice i
    JOIN   InvoiceLineItem li ON i.invoice_id = li.invoice_id
    WHERE  i.is_cancellation = 0
""", conn)
print("Order Count and Average Order Value:")
q2

Order Count and Average Order Value:


,total_orders,avg_order_value_gbp
0,36639,475.97


In [30]:
q3 = pd.read_sql_query("""
    SELECT
        p.stock_code,
        p.description,
        ROUND(SUM(li.line_total), 2)  AS total_revenue_gbp,
        SUM(li.quantity)               AS total_units_sold
    FROM   InvoiceLineItem li
    JOIN   Product p ON li.stock_code = p.stock_code
    JOIN   Invoice i ON li.invoice_id = i.invoice_id
    WHERE  i.is_cancellation = 0
      AND  li.quantity > 0
    GROUP  BY p.stock_code, p.description
    ORDER  BY total_revenue_gbp DESC
    LIMIT  10
""", conn)
print("Top 10 Products by Revenue:")
q3

Top 10 Products by Revenue:


,stock_code,description,total_revenue_gbp,total_units_sold
0,22423,REGENCY CAKESTAND 3 TIER,286486.30,24899
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,252227.81,93697
2,85099B,JUMBO BAG RED RETROSPOT,170616.68,94983
3,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
4,84879,ASSORTED COLOUR BIRD ORNAMENT,127074.17,79913
5,47566,PARTY BUNTING,103880.23,23607
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,79594.33,29477
8,79321,CHILLI LIGHTS,72860.14,15735
9,22386,JUMBO BAG PINK POLKADOT,68439.19,37711


In [31]:
q4 = pd.read_sql_query("""
    SELECT
        strftime('%Y-%m', i.invoice_date)   AS year_month,
        ROUND(SUM(li.line_total), 2)         AS monthly_revenue_gbp,
        COUNT(DISTINCT i.invoice_id)         AS order_count
    FROM   Invoice i
    JOIN   InvoiceLineItem li ON i.invoice_id = li.invoice_id
    WHERE  i.is_cancellation = 0
    GROUP  BY year_month
    ORDER  BY year_month
""", conn)
print("Monthly Revenue Trend:")
q4

Monthly Revenue Trend:


,year_month,monthly_revenue_gbp,order_count
0,2009-12,681179.72,1502
1,2010-01,538857.89,995
2,2010-02,499749.68,1093
3,2010-03,668577.19,1509
4,2010-04,587776.27,1316
5,2010-05,594886.54,1365
6,2010-06,632235.90,1482
7,2010-07,583388.53,1362
8,2010-08,596579.22,1276
9,2010-09,808145.06,1659


In [32]:
q5 = pd.read_sql_query("""
    SELECT
        c.country,
        ROUND(SUM(li.line_total), 2)   AS total_revenue_gbp,
        COUNT(DISTINCT i.invoice_id)   AS order_count,
        COUNT(DISTINCT c.customer_id)  AS unique_customers
    FROM   Invoice i
    JOIN   InvoiceLineItem li ON i.invoice_id  = li.invoice_id
    JOIN   Customer c          ON i.customer_id = c.customer_id
    WHERE  i.is_cancellation = 0
    GROUP  BY c.country
    ORDER  BY total_revenue_gbp DESC
    LIMIT  15
""", conn)
print("Revenue by Country (Top 15):")
q5

Revenue by Country (Top 15):


,country,total_revenue_gbp,order_count,unique_customers
0,United Kingdom,14623755.65,33382,5335
1,EIRE,593656.85,541,5
2,Netherlands,549952.66,216,22
3,Germany,389753.97,757,107
4,France,313838.32,587,92
5,Australia,162248.82,74,14
6,Spain,98344.32,144,39
7,Switzerland,92127.34,78,21
8,Sweden,86079.04,98,19
9,Denmark,73847.65,55,11


In [33]:
q6 = pd.read_sql_query("""
    SELECT
        c.customer_id,
        c.country,
        ROUND(SUM(li.line_total), 2)   AS total_spend_gbp,
        COUNT(DISTINCT i.invoice_id)   AS total_orders
    FROM   Customer c
    JOIN   Invoice i          ON c.customer_id = i.customer_id
    JOIN   InvoiceLineItem li ON i.invoice_id  = li.invoice_id
    WHERE  i.is_cancellation = 0
    GROUP  BY c.customer_id, c.country
    ORDER  BY total_spend_gbp DESC
    LIMIT  10
""", conn)
print("Top 10 Customers by Lifetime Spend:")
q6

Top 10 Customers by Lifetime Spend:


,customer_id,country,total_spend_gbp,total_orders
0,18102,United Kingdom,608821.65,145
1,14646,Netherlands,526751.52,145
2,14156,EIRE,304117.92,150
3,14911,EIRE,277202.97,376
4,17450,United Kingdom,246973.09,51
5,13694,United Kingdom,196482.81,143
6,17511,United Kingdom,175603.55,60
7,16446,United Kingdom,168472.50,2
8,16684,United Kingdom,147142.77,55
9,12415,Australia,144033.37,24


In [34]:

q7 = pd.read_sql_query("""
    SELECT
        SUM(is_cancellation)                                    AS cancelled_orders,
        COUNT(*)                                                AS total_invoices,
        ROUND(100.0 * SUM(is_cancellation) / COUNT(*), 2)       AS cancellation_rate_pct
    FROM Invoice
""", conn)
print("Cancellation Rate:")
q7

Cancellation Rate:


,cancelled_orders,total_invoices,cancellation_rate_pct
0,7315,43954,16.64


In [35]:
q8 = pd.read_sql_query("""
    SELECT
        CASE WHEN order_count = 1 THEN 'One-Time Buyer'
             ELSE                      'Repeat Buyer' END   AS customer_type,
        COUNT(*)                                            AS customer_count
    FROM (
        SELECT customer_id, COUNT(DISTINCT invoice_id) AS order_count
        FROM   Invoice
        WHERE  is_cancellation = 0
        GROUP  BY customer_id
    )
    GROUP  BY customer_type
    ORDER  BY customer_count DESC
""", conn)
print("Repeat vs. One-Time Customers:")
q8

Repeat vs. One-Time Customers:


,customer_type,customer_count
0,Repeat Buyer,4236
1,One-Time Buyer,1625


In [36]:
q9 = pd.read_sql_query("""
    SELECT
        strftime('%Y', i.invoice_date)    AS year,
        ROUND(SUM(li.line_total), 2)       AS annual_revenue_gbp,
        COUNT(DISTINCT i.invoice_id)       AS total_orders,
        COUNT(DISTINCT i.customer_id)      AS unique_customers
    FROM   Invoice i
    JOIN   InvoiceLineItem li ON i.invoice_id = li.invoice_id
    WHERE  i.is_cancellation = 0
    GROUP  BY year
    ORDER  BY year
""", conn)
print("Year-over-Year Revenue:")
q9

Year-over-Year Revenue:


,year,annual_revenue_gbp,total_orders,unique_customers
0,2009,681179.72,1502,952
1,2010,8564189.90,18129,4215
2,2011,8193575.93,17008,4214


In [37]:
q10 = pd.read_sql_query("""
    SELECT
        p.stock_code,
        p.description,
        SUM(li.quantity)               AS total_units_sold,
        COUNT(DISTINCT li.invoice_id)  AS times_ordered
    FROM   InvoiceLineItem li
    JOIN   Product p ON li.stock_code = p.stock_code
    JOIN   Invoice i ON li.invoice_id = i.invoice_id
    WHERE  i.is_cancellation = 0
      AND  li.quantity > 0
    GROUP  BY p.stock_code, p.description
    ORDER  BY total_units_sold DESC
    LIMIT  10
""", conn)
print("Top 10 Products by Units Sold:")
q10

Top 10 Products by Units Sold:


,stock_code,description,total_units_sold,times_ordered
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,109169,920
1,85099B,JUMBO BAG RED RETROSPOT,94983,3260
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,93697,4895
3,21212,PACK OF 72 RETROSPOT CAKE CASES,91263,2508
4,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,1
5,84879,ASSORTED COLOUR BIRD ORNAMENT,79913,2652
6,22197,SMALL POPCORN HOLDER,77971,1752
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,195
8,17003,BROCADE RING PURSE,71129,387
9,21977,PACK OF 60 PINK PAISLEY CAKE CASES,55270,1578


In [38]:
conn.close()